In [137]:
import sqlparse
from sqlparse.sql import IdentifierList, Identifier
from sqlparse.tokens import Keyword, DML
import os
import pandas as pd
from collections import defaultdict

from IPython.display import HTML
from IPython.display import display_html


%load_ext autoreload
%autoreload 2

import sys
sys.path.append("..")
from utils.utils import *

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [138]:
import networkx as nx
from networkx.readwrite import json_graph
import pickle

def load_sql_rep(fn, dummy=None):
    assert ".pkl" in fn
    try:
        with open(fn, "rb") as f:
            query = pickle.load(f)
    except Exception as e:
        print(e)
        print(fn + " failed to load...")
        exit(-1)

    query["subset_graph"] = nx.OrderedDiGraph(json_graph.adjacency_graph(query["subset_graph"]))
    query["join_graph"] = json_graph.adjacency_graph(query["join_graph"])
    if "subset_graph_paths" in query:
        query["subset_graph_paths"] = nx.OrderedDiGraph(json_graph.adjacency_graph(query["subset_graph_paths"]))

    return query

In [139]:
SQL_DIR = "multi_column_1/"

In [140]:
WK = "ceb"

# QDIR = "./imdb-sqls/" + SQL_DIR
# OUTPATH = os.path.join("./imdb-dfs/" + SQL_DIR)

QDIR = "./imdb-new-sqls/" + SQL_DIR
OUTPATH = os.path.join("./imdb-new-dfs/" + SQL_DIR)

make_dir(OUTPATH)

COLS_TO_TABLES = {}


In [141]:
qdict = {}
qfns = []

# QREP_DIR = './imdb-workload/' + SQL_DIR
QREP_DIR = './imdb-new-workload/' + SQL_DIR


if WK in ["job", "ceb"]:
    qfns = os.listdir(QREP_DIR)
    for fn in qfns:
        qfn = os.path.join(QREP_DIR, fn)
        if ".pkl" in qfn:
            qdict[fn] = load_sql_rep(qfn)

In [142]:
FNS = os.listdir(QDIR)

In [143]:
sqls = []
qreps = []

for fi, fn in enumerate(FNS):
    if ".sql" not in fn:
        continue
    if "all" in fn:
        continue
    
    fnkey = fn.replace(".sql", ".pkl")
    qreps.append(qdict[fnkey])
        
    fn = os.path.join(QDIR, fn)
    with open(fn, "r", encoding='utf-8') as f:
        sql = f.read()

    sql = sql.replace("IN", "in")
    sql = sql.replace("In", "in")
    sql = sql.replace("ILIKE", "like")
    sql = sql.replace("LIKE", "like")
    
    if "return_rank" in sql:
        print(sql)
    sqls.append(sql)

In [144]:
print(len(sqls), len(qreps))

1 1


In [145]:
for i, sql in enumerate(sqls):
    sql = sql.replace("IN", "in")
    sql = sql.replace("In", "in")
    sql = sql.replace("ILIKE", "like")
    sql = sql.replace("LIKE", "like")
    sqls[i] = sql

In [146]:
len(sqls)

1

In [147]:
#opdf,exprdf = parse_sqls_single([sql], 0, COLS_TO_TABLES)
#opdf
parse_sql_preds(sql)

,op,dtype,optype,constant,input,column,opstring,is_id
0,LIKE,unknown,contains,'%lise%',n,name,n.name like '%lise%',0


In [148]:
start = time.time()
opdf,exprdf = parse_sqls_par(sqls, COLS_TO_TABLES)
print("parse sqls took(s): ", int(time.time()-start))
print("Num Ops: ", len(opdf), "Num Expressions: ", len(exprdf))

number of sql groups:  1
num_par is  1 

par_args is 
 [(["SELECT n.name, mi1.info, Min(t.production_year), MAX(t.production_year)\nFROM title as t,\nkind_type as kt,\nmovie_info as mi1,\ninfo_type as it1,\ncast_info as ci,\nrole_type as rt,\nname as n\nWHERE\nt.id = ci.movie_id\nAND t.id = mi1.movie_id\nAND mi1.info_type_id = it1.id\nAND t.kind_id = kt.id\nAND ci.person_id = n.id\nAND ci.movie_id = mi1.movie_id\nAND ci.role_id = rt.id\nAND (it1.id in ('4'))\nAND (mi1.info in ('Danish','Spanish'))\nAND (n.name like '%lise%')\nAND (kt.kind in ('tv series','video game','video movie'))\nAND (rt.role in ('director','editor','miscellaneous crew','producer','production designer'))\nGROUP BY mi1.info, n.name\n"], 0, {})]
here parse_sqls_single
allpreds is 
 [     op    dtype    optype  constant input column              opstring  \
0  LIKE  unknown  contains  '%lise%'     n   name  n.name like '%lise%'   

   is_id  jobid  
0      0      0  ]
opdf is 
      op    dtype    optype  constant inp

In [149]:
print("len(qreps) is ",len(qreps))
if len(qreps) != 0:
#     for jid in set(exprdf["jobid"]):
#         qrep = qreps[int(jid)]
#         sql = sqls[int(jid)]
#         assert qrep["sql"].lower() == sql.lower()
#         tmp = exprdf[exprdf.jobid == jid]

    rowcounts = []
    totals = []
    sels = []
    filtersqls = []
    
    for _,row in exprdf.iterrows():
        jid = row["jobid"]
        qrep = qreps[int(jid)]
        sql = sqls[int(jid)]
        try:
            cards = qrep["subset_graph"].nodes()[tuple([row["alias"],])]["cardinality"]
            rowcounts.append(cards["actual"])
            totals.append(cards["total"])
            sels.append(min(float(cards["actual"]) / cards["total"], 1.0))
            
            filtersql = row["filtersql"]
            real_name = qrep["join_graph"].nodes()[row["alias"]]["real_name"]
            filtersql = filtersql.replace(row["input"], real_name, 1)
            filtersqls.append(filtersql)
        except:
            rowcounts.append(-1)
            totals.append(-1)
            sels.append(-1)
            filtersqls.append(-1)
        

len(qreps) is  1


In [150]:
if len(qreps) == 0:
    exprdf["InputCardinality"] = -1
    exprdf["RowCount"] = -1
    exprdf["Selectivity"] = -1
else:
    exprdf["InputCardinality"] = totals
    exprdf["Selectivity"] = sels
    exprdf["RowCount"] = rowcounts
    exprdf["filtersql"] = filtersqls

In [151]:
print(len(set(opdf["exprhash"])))
opdf.head(5)

1


,op,dtype,optype,constant,input,column,opstring,is_id,jobid,discrete_ops,cont_ops,like_ops,alias,exprhash
0,LIKE,unknown,contains,'%lise%',n,name,n.name like '%lise%',0,0,0,0,1,n,1412706111350165054529954383171903240337882399016


In [152]:
print(len(set(exprdf["exprhash"])))
exprdf.head(5)

1


,filtersql,jobid,input,alias,exprhash,unique_filter_cols,discrete_ops,cont_ops,like_ops,is_id,num_unique_ops,num_ops,num_discrete_consts,num_likes,like_lens,InputCardinality,Selectivity,RowCount
0,SELECT COUNT(*) from name AS n WHERE n.name li...,0,n,n,1412706111350165054529954383171903240337882399016,1,0,0,1,0,1,1,0,1,8,1736455,0.001572,2729


In [153]:
print(len(exprdf))
print(len(set(exprdf["jobid"])))

1
1


In [154]:
print(len(sqls))

1


In [155]:
exprdf["Selectivity"].describe()

count    1.000000
mean     0.001572
std           NaN
min      0.001572
25%      0.001572
50%      0.001572
75%      0.001572
max      0.001572
Name: Selectivity, dtype: float64

In [156]:
print(len(opdf))
edf = exprdf[["exprhash", "InputCardinality", "Selectivity", "RowCount"]]
opdf = opdf.merge(edf, on="exprhash", how="inner")
print(len(opdf))
opdf.head(5)

1
1


,op,dtype,optype,constant,input,column,opstring,is_id,jobid,discrete_ops,cont_ops,like_ops,alias,exprhash,InputCardinality,Selectivity,RowCount
0,LIKE,unknown,contains,'%lise%',n,name,n.name like '%lise%',0,0,0,0,1,n,1412706111350165054529954383171903240337882399016,1736455,0.001572,2729


In [157]:
exprdf["filtersql"].values

array(["SELECT COUNT(*) from name AS n WHERE n.name like '%lise%'"],
      dtype=object)

In [158]:
if "stack" in WK:
    opdf["opstring"] = -1

In [159]:
exprdf.head(5)

,filtersql,jobid,input,alias,exprhash,unique_filter_cols,discrete_ops,cont_ops,like_ops,is_id,num_unique_ops,num_ops,num_discrete_consts,num_likes,like_lens,InputCardinality,Selectivity,RowCount
0,SELECT COUNT(*) from name AS n WHERE n.name li...,0,n,n,1412706111350165054529954383171903240337882399016,1,0,0,1,0,1,1,0,1,8,1736455,0.001572,2729


In [160]:
print(opdf["is_id"].mean())
print(exprdf["is_id"].mean())

0.0
0.0


In [161]:
set(opdf["column"])

{'name'}

In [162]:
opdf["workload"] = WK
exprdf["workload"] = WK
opdf["db"] = WK
exprdf["db"] = WK

import time
start = time.time()

opfn = os.path.join(OUTPATH, "op_df.csv")
exprfn = os.path.join(OUTPATH, "expr_df.csv")

opdf.to_csv(opfn, index=False)
print("opdf to csv took: ", time.time()-start)

exprdf.to_csv(exprfn, index=False)

opdf to csv took:  0.0011293888092041016


In [163]:
print(opfn)

./imdb-new-dfs/multi_column_1/op_df.csv
